RuntimeError: Sizes of tensors must match except in dimension 1. Expected size 10027 but got size 577967 for tensor number 1 in the list.

In [4]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch_geometric.utils import to_networkx
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from sklearn.preprocessing import LabelEncoder, MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import os

## 1. Tiền xử lý dữ liệu

In [5]:
# from google.colab import drive
# drive.mount('/content/drive')

In [6]:
# file_path = '/content/drive/My Drive/Colab Notebooks/Data/train.csv'
# train_df = pd.read_csv(file_path).sample(frac=0.8, random_state=42)
# file_path = '/content/drive/My Drive/Colab Notebooks/Data/segment_status.csv'
# segment_status_df = pd.read_csv(file_path)

train_df = pd.read_csv("../Dataset/train.csv")
segment_status_df = pd.read_csv("../Dataset/segment_status.csv")

In [7]:
# Kiểm tra số dòng, số cột của từng file
for name, df in [("segment_status_df", segment_status_df),
                 ("train_df", train_df)]:
    print(f"{name}: {df.info}")

segment_status_df: <bound method DataFrame.info of          _id                updated_at  segment_id  velocity
0          0  2020-07-03T14:55:31.869Z       24845        20
1          1  2020-07-03T15:02:56.048Z       33923        10
2          2  2020-07-04T08:15:52.696Z       33824         5
3          3  2020-07-04T08:15:59.903Z       33824         5
4          4  2020-07-04T08:16:08.201Z       33824         5
...      ...                       ...         ...       ...
90933  90933  2021-04-22T06:52:39.280Z       52247         1
90934  90934  2021-04-22T06:52:52.501Z       52247         1
90935  90935  2021-04-22T06:53:02.335Z       52247         1
90936  90936  2021-04-22T06:53:14.294Z       52247         1
90937  90937  2021-04-22T06:53:27.300Z       52247         1

[90938 rows x 4 columns]>
train_df: <bound method DataFrame.info of          _id  segment_id        date  weekday        period LOS   s_node_id  \
0          0          26  2021-04-16        4   period_0_30   A   366

In [8]:
# Gom street_type hiếm
threshold = 100
street_type_counts = train_df['street_type'].value_counts()
train_df['street_type_adjusted'] = train_df['street_type'].apply(
    lambda x: x if street_type_counts.get(x, 0) >= threshold else 'other'
)
le_street_type = LabelEncoder()
train_df['street_type_encoded'] = le_street_type.fit_transform(train_df['street_type_adjusted'].fillna('other'))

In [9]:
# Tính los_dist
los_map = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4, 'F': 5}
train_df['LOS_encoded'] = train_df['LOS'].map(los_map)
los_dist = pd.crosstab(train_df['segment_id'], train_df['LOS_encoded'], normalize='index')
los_dist.columns = [f'los_{col}_ratio' for col in los_dist.columns]
los_dist = los_dist.reindex(train_df['segment_id'].unique(), fill_value=0)

In [10]:
# Hàm convert period (vd: "period_14_30") thành timedelta
def period_to_time(period_str):
    try:
        _, hour_str, min_str = period_str.split("_")
        hour = int(hour_str)
        minute = int(min_str)
        return pd.to_timedelta(f"{hour}:{minute}:00")
    except:
        return pd.NaT

# Apply và tạo cột thời gian đầy đủ
train_df['time_delta'] = train_df['period'].apply(period_to_time)
train_df['date'] = pd.to_datetime(train_df['date']) + train_df['time_delta']

# Xoá cột phụ nếu muốn
train_df.drop(columns='time_delta', inplace=True)

# Chuyển 'date' và 'updated_at' về datetime
train_df['date'] = pd.to_datetime(train_df['date']).dt.tz_localize(None)
segment_status_df['updated_at'] = pd.to_datetime(segment_status_df['updated_at']).dt.tz_localize(None)

# Sort trước khi dùng merge_asof
train_df = train_df.sort_values(by='date')
segment_status_df = segment_status_df.sort_values(by='updated_at')

# Merge gần đúng theo thời gian, trong cùng segment_id
merged_df = pd.merge_asof(
    train_df,
    segment_status_df,
    by='segment_id',
    left_on='date',
    right_on='updated_at',
    direction='nearest',  # hoặc 'backward' nếu bạn chỉ muốn dùng dữ liệu trước đó
    tolerance=pd.Timedelta('30min')  # chỉ chấp nhận khớp nếu lệch thời gian <= 30 phút
)

# Kiểm tra kết quả merge
print("\nMerged DataFrame:")
print(merged_df.head())
# Kiểm tra các cột trong DataFrame sau khi merge
print("\nColumns in Merged DataFrame:")
print(merged_df.columns)


Merged DataFrame:
   _id_x  segment_id                date  weekday        period LOS  \
0   9598       24845 2020-07-03 14:30:00        4  period_14_30   D   
1  13056       33923 2020-07-03 15:00:00        4  period_15_00   F   
2  12977       33824 2020-07-04 08:00:00        5   period_8_00   F   
3  12978       33824 2020-07-04 08:30:00        5   period_8_30   E   
4  23199       56816 2020-07-04 08:30:00        5   period_8_30   F   

    s_node_id   e_node_id  length  street_id  ...  long_snode  lat_snode  \
0  2409410635  3771416347      73  138877181  ...  106.651961  10.793623   
1  5769406275  5769406276      78  213726584  ...  106.651548  10.787537   
2  2233730990  2299409160      67  213719139  ...  106.654482  10.785806   
3  2233730990  2299409160      67  213719139  ...  106.654482  10.785806   
4  5772558121  5772558074      89  398665528  ...  106.657161  10.778896   

   long_enode  lat_enode  street_type_adjusted  street_type_encoded  \
0  106.652559  10.793328  

In [ ]:
# Kiểm tra NaN và số bản ghi
print("\nKiểm tra NaN trong merged_df:")
print(merged_df[['length', 'velocity', 'street_level', 'LOS']].isna().sum())
print(f"Số bản ghi trong train_df: {len(train_df)}")
print(f"Số bản ghi trong segment_status_df: {len(segment_status_df)}")
print(f"Số bản ghi trong merged_df: {len(merged_df)}")


Kiểm tra NaN trong merged_df:
length          0
velocity        0
street_level    0
LOS             0
dtype: int64
Số bản ghi trong train_df: 33441
Số bản ghi trong segment_status_df: 90938
Số bản ghi trong merged_df: 33441


## 2. Chuẩn bị model

### 2.1 Tạo đặc trưng node

In [12]:
# Lọc node phổ biến
node_counts = pd.concat([merged_df['s_node_id'], merged_df['e_node_id']]).value_counts()
top_nodes = node_counts.head(500).index
merged_df = merged_df[merged_df['s_node_id'].isin(top_nodes) & merged_df['e_node_id'].isin(top_nodes)]
print(f"Filtered dataset to {len(merged_df)} rows with {len(top_nodes)} nodes")

Filtered dataset to 7228 rows with 500 nodes


### 2.2 RandomForest để dự đoán các đặc trưng

In [14]:
from imblearn.over_sampling import SMOTE
from collections import Counter

# Merge los_dist vào merged_df
merged_df = merged_df.merge(los_dist, on='segment_id', how='left').fillna(0)
# Chuẩn bị dữ liệu
scaler = StandardScaler()
node_freq = pd.concat([merged_df['s_node_id'], merged_df['e_node_id']]).value_counts()
features = merged_df[['length', 'velocity', 'street_type_encoded', 'los_0_ratio', 'los_1_ratio', 'los_2_ratio', 'los_3_ratio', 'los_4_ratio', 'los_5_ratio']].copy()
features['degree_s'] = merged_df['s_node_id'].map(node_freq)
features['degree_e'] = merged_df['e_node_id'].map(node_freq)
features = scaler.fit_transform(features)

# Mã hóa street_level
street_level_encoder = LabelEncoder()
merged_df['street_level'] = street_level_encoder.fit_transform(merged_df['street_level'])

# Random Forest với SMOTE
def train_feature_predictors(features, targets):
    X_train, X_test, y_train_length, y_test_length = train_test_split(features, targets['length'], test_size=0.2, random_state=42)
    _, _, y_train_velocity, y_test_velocity = train_test_split(features, targets['velocity'], test_size=0.2, random_state=42)
    _, _, y_train_street_level, y_test_street_level = train_test_split(features, targets['street_level'], test_size=0.2, random_state=42)
    _, _, y_train_los, y_test_los = train_test_split(features, targets['LOS'], test_size=0.2, random_state=42)

    smote = SMOTE(random_state=42,k_neighbors=3, sampling_strategy={'A': 5000, 'B': 5000, 'C': 5000, 'D': 5000, 'E': 5000, 'F': 5000})
    X_train_los, y_train_los = smote.fit_resample(X_train, y_train_los)
    print("\nPhân bố LOS sau SMOTE:")
    print(Counter(y_train_los))

    length_clf = RandomForestRegressor(n_estimators=100, random_state=42)
    length_clf.fit(X_train, y_train_length)
    velocity_clf = RandomForestRegressor(n_estimators=100, random_state=42)
    velocity_clf.fit(X_train, y_train_velocity)
    street_level_clf = RandomForestClassifier(n_estimators=100, random_state=42)
    street_level_clf.fit(X_train, y_train_street_level)
    street_level_pred = street_level_clf.predict(X_test)
    street_level_acc = accuracy_score(y_test_street_level, street_level_pred)

    class_weights = {'A': 5.8, 'B': 5.2, 'C': 5.2, 'D': 4.0, 'E': 5.0, 'F': 6.2}
    los_clf = RandomForestClassifier(n_estimators=100, random_state=42, class_weight=class_weights)
    los_clf.fit(X_train_los, y_train_los)
    los_pred = los_clf.predict(X_test)
    los_acc = accuracy_score(y_test_los, los_pred)

    print("\nRandom Forest Evaluation:")
    print(f"Street Level Accuracy: {street_level_acc:.4f}")
    print(f"LOS Accuracy: {los_acc:.4f}")

    return length_clf, velocity_clf, street_level_clf, los_clf, features.shape[1]

targets = {
    'length': merged_df['length'].values,
    'velocity': merged_df['velocity'].values,
    'street_level': merged_df['street_level'].values,
    'LOS': merged_df['LOS'].values
}

length_clf, velocity_clf, street_level_clf, los_clf, input_dim = train_feature_predictors(features, targets)


Phân bố LOS sau SMOTE:
Counter({'F': 5000, 'A': 5000, 'C': 5000, 'E': 5000, 'D': 5000, 'B': 5000})

Random Forest Evaluation:
Street Level Accuracy: 1.0000
LOS Accuracy: 0.9184


### 2.3 Hàm gán các đặc trưng

In [15]:
# Gán đặc trưng
def assign_edge_features(G, length_clf, velocity_clf, street_level_clf, los_clf, scaler, street_level_encoder):
    los_predictions = []
    for u, v in G.edges():
        degree_u = G.degree(u)
        degree_v = G.degree(v)
        length_sample = np.random.normal(merged_df['length'].mean(), merged_df['length'].std() * 0.5)
        velocity_sample = np.random.normal(merged_df['velocity'].mean(), merged_df['velocity'].std() * 0.5)
        street_type_sample = np.random.choice(merged_df['street_type_encoded'])
        los_ratios = np.random.normal(
            merged_df[['los_0_ratio', 'los_1_ratio', 'los_2_ratio', 'los_3_ratio', 'los_4_ratio', 'los_5_ratio']].mean(),
            merged_df[['los_0_ratio', 'los_1_ratio', 'los_2_ratio', 'los_3_ratio', 'los_4_ratio', 'los_5_ratio']].std() * 0.5
        )
        feature_input = np.array([[length_sample, velocity_sample, street_type_sample, *los_ratios, degree_u, degree_v]])
        feature_input = scaler.transform(feature_input)

        length = length_clf.predict(feature_input)[0]
        velocity = velocity_clf.predict(feature_input)[0]
        street_level = street_level_encoder.inverse_transform([street_level_clf.predict(feature_input)[0]])[0]
        los = los_clf.predict(feature_input)[0]

        G.edges[u, v]['length'] = round(max(0, length), 1)
        G.edges[u, v]['velocity'] = round(max(0, velocity), 1)
        G.edges[u, v]['street_level'] = street_level
        G.edges[u, v]['LOS'] = los
        los_predictions.append(los)
    print("\nPhân bố LOS dự đoán trong assign_edge_features:")
    print(Counter(los_predictions))
    return G

## 3. Chuẩn bị model

In [16]:
# Chuẩn bị đồ thị
def create_graphs(df):
    dates = df['date'].dt.date.unique()[:60] # lấy 60 ngày đầu
    graphs = []
    node_encoder = LabelEncoder()
    all_nodes = pd.concat([df['s_node_id'], df['e_node_id']]).unique()
    node_encoder.fit(all_nodes)

    for date in dates:
        daily_df = df[df['date'].dt.date == date]
        G = nx.Graph()
        G.graph['date'] = date
        for _, row in daily_df.iterrows():
            s_node = node_encoder.transform([row['s_node_id']])[0]
            e_node = node_encoder.transform([row['e_node_id']])[0]
            G.add_edge(s_node, e_node, segment_id=row['segment_id'])
        if G.number_of_edges() >= 10:
            graphs.append(G)
            print(f"Graph for {date}: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
    return graphs, node_encoder

graphs, node_encoder = create_graphs(merged_df)
if not graphs:
    print("Error: No graphs created. Try reducing edge threshold.")
    exit()
max_nodes = min(max(g.number_of_nodes() for g in graphs), 500)  # Giới hạn max_nodes
print(f"Created {len(graphs)} graphs with max nodes: {max_nodes}")

Graph for 2020-08-02: 73 nodes, 58 edges
Graph for 2020-08-03: 73 nodes, 58 edges
Graph for 2020-11-16: 39 nodes, 22 edges
Graph for 2020-11-17: 38 nodes, 23 edges
Graph for 2020-11-18: 58 nodes, 34 edges
Graph for 2020-11-19: 55 nodes, 29 edges
Graph for 2020-11-20: 39 nodes, 23 edges
Graph for 2020-11-26: 94 nodes, 55 edges
Graph for 2020-11-27: 21 nodes, 11 edges
Graph for 2020-11-28: 137 nodes, 78 edges
Graph for 2020-11-29: 309 nodes, 200 edges
Graph for 2020-11-30: 298 nodes, 191 edges
Graph for 2020-12-01: 316 nodes, 203 edges
Graph for 2020-12-02: 307 nodes, 186 edges
Graph for 2020-12-03: 72 nodes, 45 edges
Graph for 2020-12-05: 232 nodes, 134 edges
Graph for 2020-12-06: 276 nodes, 179 edges
Graph for 2020-12-07: 250 nodes, 146 edges
Graph for 2020-12-09: 96 nodes, 51 edges
Graph for 2020-12-10: 291 nodes, 176 edges
Graph for 2020-12-12: 276 nodes, 188 edges
Graph for 2020-12-13: 242 nodes, 150 edges
Graph for 2020-12-15: 34 nodes, 19 edges
Graph for 2020-12-18: 35 nodes, 22 e

In [17]:
# Chuẩn bị sparse tensor
adj_tensors = []
for g in graphs:
    adj = nx.adjacency_matrix(g).tocoo()
    indices = torch.tensor([adj.row, adj.col], dtype=torch.long)
    values = torch.tensor(adj.data, dtype=torch.float32)
    adj_tensor = torch.sparse_coo_tensor(indices, values, size=(max_nodes, max_nodes))
    adj_tensors.append(adj_tensor.to_dense())
adj_tensor = torch.stack(adj_tensors)
print(f"Adjacency tensor shape: {adj_tensor.shape}")

Adjacency tensor shape: torch.Size([33, 316, 316])


/tmp/ipykernel_9195/3604917178.py:5: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:254.)
  indices = torch.tensor([adj.row, adj.col], dtype=torch.long)


## 4. Định nghĩa model

In [18]:
# Mô hình GraphRNN
class GraphRNN(nn.Module):
    def __init__(self, max_nodes, hidden_dim):
        super(GraphRNN, self).__init__()
        self.max_nodes = max_nodes
        self.hidden_dim = hidden_dim
        self.node_rnn = nn.GRU(input_size=1, hidden_size=hidden_dim, batch_first=True)
        self.edge_rnn = nn.GRU(input_size=hidden_dim, hidden_size=hidden_dim, batch_first=True)
        self.output_layer = nn.Linear(hidden_dim, max_nodes)
        self.dropout = nn.Dropout(0.7)

    def forward(self, adj_matrix):
        batch_size = adj_matrix.size(0)
        node_hidden = torch.zeros(1, batch_size, self.hidden_dim).to(adj_matrix.device)
        node_inputs = torch.ones(batch_size, self.max_nodes, 1).to(adj_matrix.device)
        node_outputs, _ = self.node_rnn(node_inputs, node_hidden)
        node_outputs = self.dropout(node_outputs)

        edge_probs = []
        for i in range(self.max_nodes):
            node_i = node_outputs[:, i, :].unsqueeze(1)
            edge_output, node_hidden = self.edge_rnn(node_i, node_hidden)
            edge_prob = torch.sigmoid(self.output_layer(edge_output.squeeze(1)))
            edge_probs.append(edge_prob)

        edge_probs = torch.stack(edge_probs, dim=1)
        return edge_probs

## 5. Huấn luyện model

In [19]:
# Khởi tạo và huấn luyện
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = GraphRNN(max_nodes=max_nodes, hidden_dim=64).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.0001)
criterion = nn.BCELoss()

model.train()
adj_tensor = adj_tensor.to(device)
losses = []
for epoch in range(80):
    optimizer.zero_grad()
    output = model(adj_tensor)
    loss = criterion(output, adj_tensor)
    loss.backward()
    optimizer.step()
    losses.append(loss.item())
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 10, Loss: 0.6882
Epoch 20, Loss: 0.6828
Epoch 30, Loss: 0.6763
Epoch 40, Loss: 0.6684
Epoch 50, Loss: 0.6579
Epoch 60, Loss: 0.6438
Epoch 70, Loss: 0.6242
Epoch 80, Loss: 0.5963


In [20]:
# Đánh giá đồ thị
def evaluate_graph(generated_graph, real_graphs):
    print("\nGraph Evaluation:")
    print(f"Generated graph: {generated_graph.number_of_nodes()} nodes, {generated_graph.number_of_edges()} edges")
    avg_nodes = np.mean([g.number_of_nodes() for g in real_graphs])
    avg_edges = np.mean([g.number_of_edges() for g in real_graphs])
    print(f"Average real graph: {avg_nodes:.1f} nodes, {avg_edges:.1f} edges")
    gen_degrees = [d for _, d in generated_graph.degree()]
    real_degrees = [d for g in real_graphs for _, d in g.degree()]
    print(f"Generated degree mean: {np.mean(gen_degrees):.2f}, std: {np.std(gen_degrees):.2f}")
    print(f"Real degree mean: {np.mean(real_degrees):.2f}, std: {np.std(real_degrees):.2f}")
    gen_clustering = nx.average_clustering(generated_graph)
    real_clustering = np.mean([nx.average_clustering(g) for g in real_graphs])
    print(f"Generated clustering coefficient: {gen_clustering:.4f}")
    print(f"Average real clustering coefficient: {real_clustering:.4f}")

In [21]:
# Sinh đồ thị
model.eval()
with torch.no_grad():
    generated_adj = model(torch.zeros(1, max_nodes, max_nodes).to(device))
    generated_adj = generated_adj.squeeze(0).cpu().numpy()

    num_edges = 100
    edge_indices = np.argsort(generated_adj.ravel())[-num_edges*2:]
    rows, cols = np.unravel_index(edge_indices, generated_adj.shape)
    edges = [(r, c) for r, c in zip(rows, cols) if r < c and generated_adj[r, c] > 0.1]

    generated_graph = nx.Graph()
    generated_graph.add_nodes_from(range(max_nodes))
    generated_graph.add_edges_from(edges)

    generated_graph = assign_edge_features(
        generated_graph, length_clf, velocity_clf, street_level_clf, los_clf,
        scaler, street_level_encoder
    )

    edges = [(u, int(v)) for u, v in generated_graph.edges()]
    print(f"\nĐồ thị: {edges}")
    print("Edge features:")
    for u, v in edges:
        length = generated_graph.edges[u, v]['length']
        velocity = generated_graph.edges[u, v]['velocity']
        street_level = generated_graph.edges[u, v]['street_level']
        los = generated_graph.edges[u, v]['LOS']
        print(f"- Edge ({u}, {v}): length={length}, velocity={velocity}, street_level={street_level}, LOS={los}")

    os.makedirs('generated_graphs', exist_ok=True)
    nx.write_edgelist(generated_graph, 'generated_graphs/generated_graph.edgelist')

    evaluate_graph(generated_graph, graphs)

/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklea


Phân bố LOS dự đoán trong assign_edge_features:
Counter({'A': 30, 'B': 9, 'C': 8, 'D': 1})

Đồ thị: [(18, 106), (19, 106), (20, 106), (44, 106), (45, 106), (46, 106), (47, 106), (48, 106), (49, 106), (50, 106), (51, 106), (52, 106), (53, 106), (54, 106), (55, 106), (56, 106), (57, 106), (58, 106), (59, 106), (60, 106), (61, 106), (62, 106), (63, 106), (64, 106), (65, 106), (66, 106), (68, 106), (69, 106), (70, 106), (71, 106), (72, 106), (73, 106), (74, 106), (75, 106), (76, 106), (77, 106), (78, 106), (79, 106), (80, 106), (81, 106), (82, 106), (83, 106), (84, 106), (85, 106), (86, 106), (87, 106), (88, 106), (89, 106)]
Edge features:
- Edge (18, 106): length=56.0, velocity=32.0, street_level=4, LOS=B
- Edge (19, 106): length=49.0, velocity=46.0, street_level=4, LOS=A
- Edge (20, 106): length=23.0, velocity=40.0, street_level=2, LOS=A
- Edge (44, 106): length=101.8, velocity=22.0, street_level=4, LOS=D
- Edge (45, 106): length=51.0, velocity=49.0, street_level=2, LOS=A
- Edge (46, 10

/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(
/home/le-minh-hoang/Learning/PythonProjects/myenv/lib/python3.12/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but StandardScaler was fitted with feature names
  warnings.warn(


## 6. Đánh giá đồ thị